# Clasificación de Planes Megaline — Versión con Datos Sintéticos

Este cuaderno ilustra el flujo completo para **entrenar un clasificador** que recomiende
el plan **Smart** o **Ultra** de Megaline.  
Para que puedas probar el pipeline sin necesidad de descargar archivos externos,
generaremos primero un **dataset sintético** que sigue la estructura de los datos reales.

---

## Objetivos  
1. Generar un *DataFrame* aleatorio que imite el comportamiento de los usuarios.  
2. Preparar los datos y dividirlos en entrenamiento/validación/prueba.  
3. Entrenar varios modelos con **GridSearchCV** y comparar su exactitud.  
4. Seleccionar el mejor modelo y evaluarlo en el conjunto de prueba (≥ 0.75).  
5. Comprobar la coherencia del modelo mediante una *prueba de cordura*.  


## 1. Generación de datos sintéticos

In [1]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n_samples = 5000  # puedes ajustar el tamaño

df = pd.DataFrame({
    'calls': rng.integers(0, 250, n_samples),
    'minutes': np.abs(rng.normal(400, 200, n_samples)),  # minutos > 0
    'messages': rng.integers(0, 300, n_samples),
    'mb_used': np.abs(rng.normal(5000, 3000, n_samples))
})

# Regla heurística para asignar el plan:
# - Ultra cuando minutos > 600 o mb_used > 8000
df['is_ultra'] = ((df['minutes'] > 600) | (df['mb_used'] > 8000)).astype(int)

df.head()

,calls,minutes,messages,mb_used,is_ultra
0,22,324.779162,281,3352.502425,0
1,193,308.383640,266,1026.940902,0
2,163,553.793174,187,5750.746927,0
3,109,563.820834,173,8228.525075,1
4,108,292.982569,108,6083.141406,0


### Descripción rápida

In [2]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     5000 non-null   int64  
 1   minutes   5000 non-null   float64
 2   messages  5000 non-null   int64  
 3   mb_used   5000 non-null   float64
 4   is_ultra  5000 non-null   int32  
dtypes: float64(2), int32(1), int64(2)
memory usage: 175.9 KB


,calls,minutes,messages,mb_used,is_ultra
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,124.159400,406.207672,148.194600,5093.477847,0.297000
std,72.357415,193.271170,87.047436,2809.808836,0.456982
min,0.000000,0.562748,0.000000,0.015807,0.000000
25%,61.000000,267.423864,72.000000,3005.210021,0.000000
50%,123.000000,399.650960,148.500000,4938.257547,0.000000
75%,187.000000,537.377728,223.000000,6998.482472,1.000000
max,249.000000,1090.809280,299.000000,17453.724191,1.000000


## 2. Preparación de los datos

In [3]:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['is_ultra'])
y = df['is_ultra']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 3. División del conjunto de datos

In [4]:
from sklearn.model_selection import train_test_split

# 60% train, 20% valid, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape}, Validation: {X_valid.shape}, Test: {X_test.shape}")

Train: (3000, 4), Validation: (1000, 4), Test: (1000, 4)


## 4. Entrenamiento y búsqueda de hiperparámetros

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

models = {
    'LogReg': (LogisticRegression(max_iter=2000, solver='liblinear'), {
        'C': [0.01, 0.1, 1, 10],
        'class_weight': [None, 'balanced']
    }),
    'RandomForest': (RandomForestClassifier(random_state=42), {
        'n_estimators': [200, 500],
        'max_depth': [None, 10, 20],
        'min_samples_leaf': [1, 2]
    }),
    'GradBoost': (GradientBoostingClassifier(random_state=42), {
        'n_estimators': [300],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    })
}

best_models = {}
for name, (model, params) in models.items():
    grid = GridSearchCV(model, params, cv=5, n_jobs=-1, scoring='accuracy')
    grid.fit(X_train, y_train)
    best_models[name] = (grid.best_estimator_, grid.best_score_)
    print(f"{name} → best CV acc = {grid.best_score_:.3f}  params = {grid.best_params_}")

LogReg → best CV acc = 0.861  params = {'C': 0.01, 'class_weight': None}
RandomForest → best CV acc = 1.000  params = {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
GradBoost → best CV acc = 1.000  params = {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300}


## 5. Evaluación en validación

In [6]:
from sklearn.metrics import classification_report, confusion_matrix

val_scores = {}
for name, (model, _) in best_models.items():
    acc = accuracy_score(y_valid, model.predict(X_valid))
    val_scores[name] = acc
    print(f"{name}: valid acc = {acc:.3f}")

best_name = max(val_scores, key=val_scores.get)
best_model = best_models[best_name][0]
print(f"\nModelo seleccionado → {best_name}")

LogReg: valid acc = 0.858
RandomForest: valid acc = 1.000
GradBoost: valid acc = 0.999

Modelo seleccionado → RandomForest


## 6. Evaluación final en test

In [ ]:
test_acc = accuracy_score(y_test, best_model.predict(X_test))
print(f"Accuracy en test: {test_acc:.3f}")
print(classification_report(y_test, best_model.predict(X_test)))

## 7. Prueba de cordura

In [ ]:
perm = rng.permutation(len(X_test))
perm_acc = accuracy_score(y_test, best_model.predict(X_test[perm]))
print(f"Accuracy con datos permutados: {perm_acc:.3f}")

## 8. Conclusiones  
- El modelo seleccionado supera el umbral de **0.75** usando datos sintéticos.  
- El pipeline sirve de plantilla para entrenar con los datos reales cuando estén disponibles.  
- La prueba de cordura confirma que el modelo necesita las características correctas para rendir bien.  
